# M3 profile - Salmon lice pipeline

Investigation owner for measurements: Cagatay Oruk. This notebook reads the M2 database only.
Run all cells after `python src/load.py`; no clean tables, API access or raw file access is required.
All computations use the Python standard library. Outputs below were actually executed.
Category inventories are evidence for Egemen's later investigation, not completed mapping decisions.


In [1]:
from pathlib import Path
import json
import sys
import sqlite3
from collections import Counter
from statistics import median

ROOT = Path.cwd()
if not (ROOT / "src/db.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "src/db.py").exists():
    raise RuntimeError("Run from repository root or notebooks directory")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "notebooks"))
from db import DB_PATH, connect, snapshot
from profile_quick import profile, MEASUREMENTS
from clean_measurements import clean_rows
import clean_config

con = connect(DB_PATH, read_only=True)
before = snapshot(con)
p = profile(DB_PATH)
print("M2 row counts:", {table: count_hash[0] for table, count_hash in before.items()})
print("Cohort:", p["cohort_size"], "localities")
print("Foreign key violations:", p["foreign_key_violations"])


M2 row counts: {'municipality': 371, 'locality': 2710, 'lice_weekly': 6951, 'sea_temperature_weekly': 10440, 'treatment_weekly': 816, 'treatment_event': 889, 'quarantine': 0}
Cohort: 40 localities
Foreign key violations: []


## Coverage and missingness

Explicit NULLs and absent rows are different. The absent-week count below covers each locality's
first-to-last lice record only; it excludes leading/trailing unobserved weeks and does not create rows.
The most recent expected period is tied to retrieval time, not the computer's current date.


In [2]:
for table, data in p["measurements"].items():
    print("\nTABLE", table, "rows", data["rows"], "entities", data["entities"], "NULLs", data["nulls"])
    print("YEAR | ROWS | NULLS")
    for item in data["per_year"]:
        print(item["year"], item["rows"], item["nulls"], sep=" | ")
    print("LOCALITY | ROWS | NULLS | HISTORICAL NULL SHARE | FIRST | LAST | ABSENT WEEKS | FUTURE ROWS")
    for e in data["per_entity"]:
        print(e["locality_no"], e["rows"], e["nulls"], round(e["historical_null_share"], 4), e["first"], e["last"], e["absent_weeks_between_first_last"], e["future_rows"], sep=" | ")
    print("TOTAL within-span absent weeks:", sum(e["absent_weeks_between_first_last"] for e in data["per_entity"]))
    print("TOTAL historical NULLs:", sum(e["historical_nulls"] for e in data["per_entity"]))
    print("TOTAL future rows:", sum(e["future_rows"] for e in data["per_entity"]))
print("\nTemperature states (has_reported, is_null, count):", p["temperature_reporting_states"])
print("Joint historical reporting (temperature reported, lice absent, rows):", p["joint_historical_reporting"])



TABLE lice_weekly rows 6951 entities 40 NULLs 0
YEAR | ROWS | NULLS
2022 | 1503 | 0
2023 | 1550 | 0
2024 | 1352 | 0
2025 | 1599 | 0
2026 | 947 | 0
LOCALITY | ROWS | NULLS | HISTORICAL NULL SHARE | FIRST | LAST | ABSENT WEEKS | FUTURE ROWS
10110 | 197 | 0 | 0.0 | 2022-01-03 | 2026-09-28 | 51 | 0
10262 | 164 | 0 | 0.0 | 2022-05-16 | 2026-09-28 | 65 | 0
10292 | 154 | 0 | 0.0 | 2022-01-03 | 2026-09-28 | 94 | 0
10324 | 137 | 0 | 0.0 | 2022-01-03 | 2026-01-05 | 73 | 0
10328 | 165 | 0 | 0.0 | 2022-01-03 | 2026-06-29 | 70 | 0
11290 | 158 | 0 | 0.0 | 2022-01-03 | 2026-09-07 | 87 | 0
11312 | 180 | 0 | 0.0 | 2022-01-10 | 2026-09-28 | 67 | 0
11350 | 180 | 0 | 0.0 | 2022-01-03 | 2026-09-28 | 68 | 0
11405 | 138 | 0 | 0.0 | 2022-01-03 | 2026-09-28 | 110 | 0
11800 | 191 | 0 | 0.0 | 2022-01-03 | 2026-09-28 | 57 | 0
11922 | 207 | 0 | 0.0 | 2022-01-03 | 2026-08-10 | 34 | 0
12244 | 184 | 0 | 0.0 | 2022-04-11 | 2026-09-28 | 50 | 0
12652 | 139 | 0 | 0.0 | 2022-01-03 | 2025-10-06 | 58 | 0
12660 | 203 | 0 | 

## Numeric distributions and invalid-value screens

The temperature bounds -5..40 C are deliberately broad screening decisions, not provider specifications
or safe salmon temperatures. See report section 5 for source, rationale and sensitivity.
Zero is retained as a possible measurement; no zero sentinel is assumed.


In [3]:
for table, data in p["measurements"].items():
    print("\n", table)
    print("LOCALITY | MIN | MEDIAN | MAX | MOST COMMON (value,count)")
    for e in data["per_entity"]:
        print(e["locality_no"], e["min"], e["median"], e["max"], e["most_common"], sep=" | ")
    print("Screened invalid examples:", data["screening_invalid_examples"])
    print("Invalid ISO weeks:", sum(e["invalid_iso_weeks"] for e in data["per_entity"]))
    print("Future examples:", data["future_examples"])



 lice_weekly
LOCALITY | MIN | MEDIAN | MAX | MOST COMMON (value,count)
10110 | 0.0 | 0.18000000715255737 | 1.0099999904632568 | [(0.0, 19), (0.009999999776482582, 17), (0.09000000357627869, 8), (0.03999999910593033, 7), (0.18000000715255737, 7)]
10262 | 0.0 | 0.15000000596046448 | 0.48750001192092896 | [(0.0, 26), (0.05000000074505806, 5), (0.03999999910593033, 4), (0.09000000357627869, 4), (0.05999999865889549, 4)]
10292 | 0.0 | 0.08500000089406967 | 3.0799999237060547 | [(0.0, 31), (0.07999999821186066, 12), (0.12999999523162842, 10), (0.05000000074505806, 9), (0.029999999329447746, 8)]
10324 | 0.0 | 0.07000000029802322 | 1.0700000524520874 | [(0.0, 31), (0.019999999552965164, 9), (0.07000000029802322, 9), (0.1599999964237213, 7), (0.11999999731779099, 7)]
10328 | 0.0 | 0.07999999821186066 | 0.5 | [(0.0, 32), (0.019999999552965164, 12), (0.029999999329447746, 10), (0.07999999821186066, 9), (0.09000000357627869, 8)]
11290 | 0.0 | 0.05000000074505806 | 0.8999999761581421 | [(0.0, 33),

## Failures visible in sequence

Only consecutive ISO weeks count as a run or adjacent jump, including across year boundaries.
Temperature repetitions and isolated spikes are review markers, not confirmed sensor faults.
Lice runs are listed but are not automatically labelled stuck: all long exact runs here are zero.
Level-shift candidates are not automatically converted into a different unit without evidence.


In [4]:
for table, data in p["measurements"].items():
    print("\n", table, "exact runs >= 6 weeks")
    for e in data["per_entity"]:
        for run in e["runs_at_least_6"]:
            print(e["locality_no"], run)
    jumps = sorted([(j[0], e["locality_no"], j[1:]) for e in data["per_entity"] for j in e["largest_adjacent_jumps"]], reverse=True)
    print("Largest consecutive-week jumps:")
    for item in jumps[:15]:
        print(item)
    print("Persistent temperature shift candidates (4-week medians differ >=8 C):", [(e["locality_no"], candidate) for e in data["per_entity"] for candidate in e["level_shift_candidates"]])
print("Reporting conflicts:", con.execute("SELECT COUNT(*) FROM sea_temperature_weekly WHERE (has_reported=0 AND sea_temperature IS NOT NULL) OR (has_reported=1 AND sea_temperature IS NULL)").fetchone()[0])



 lice_weekly exact runs >= 6 weeks
10110 ['2025-09-15', '2025-10-20', 0.0, 6]
10262 ['2022-05-16', '2022-06-27', 0.0, 7]
10262 ['2024-04-01', '2024-06-10', 0.0, 11]
10262 ['2026-03-30', '2026-05-18', 0.0, 8]
10292 ['2022-07-04', '2022-08-15', 0.0, 7]
10324 ['2022-01-03', '2022-02-21', 0.0, 8]
10324 ['2023-03-20', '2023-05-01', 0.0, 7]
10324 ['2025-03-17', '2025-04-28', 0.0, 7]
10328 ['2023-04-03', '2023-05-29', 0.0, 9]
10328 ['2025-03-31', '2025-06-09', 0.0, 11]
11290 ['2023-05-15', '2023-07-17', 0.0, 10]
11290 ['2025-06-02', '2025-07-14', 0.0, 7]
11312 ['2022-12-12', '2023-01-16', 0.0, 6]
11312 ['2025-05-05', '2025-06-16', 0.0, 7]
11405 ['2023-05-08', '2023-06-26', 0.0, 8]
11922 ['2022-01-03', '2022-03-21', 0.0, 12]
11922 ['2022-04-11', '2022-08-22', 0.0, 20]
11922 ['2022-10-17', '2022-11-28', 0.0, 7]
11922 ['2022-12-12', '2023-02-06', 0.0, 9]
11922 ['2023-04-17', '2023-10-02', 0.0, 25]
11922 ['2023-11-06', '2024-04-01', 0.0, 22]
11922 ['2024-04-29', '2024-10-28', 0.0, 27]
11922 ['20

## Robust fences and sensitivity

Exclude invalid/missing/future observations before fitting outer IQR fences within each M2 locality.
Use 3*IQR, at least 20 valid observations, and skip zero-width IQR. Flag and retain unusual values.
If Egemen establishes identity splits or unit changes, grouping/statistics must be revisited before final M3.


In [5]:
for table, sql in MEASUREMENTS.items():
    rows = con.execute(sql).fetchall()
    kind = "lice" if table == "lice_weekly" else "temperature"
    cleaned, counts = clean_rows(rows, kind)
    print(table, "proposed mutually exclusive flags:", counts)
    print("LOCALITY | FLAGS")
    for key in sorted({r[0] for r in rows}):
        number = con.execute("SELECT locality_no FROM locality WHERE locality_key=?", (key,)).fetchone()[0]
        print(number, dict(Counter(r[5] for r in cleaned if r[0] == key)))
    original = clean_config.IQR_MULTIPLIER
    try:
        for factor in (1.5, 3.0, 4.0):
            clean_config.IQR_MULTIPLIER = factor
            _, sensitivity_counts = clean_rows(rows, kind)
            print("IQR multiplier", factor, sensitivity_counts)
    finally:
        clean_config.IQR_MULTIPLIER = original
    if kind == "temperature":
        for lo, hi in ((-3., 30.), (-5., 40.), (-10., 50.)):
            n = sum(r[3] is not None and not lo <= r[3] <= hi for r in rows)
            print("Temperature screening sensitivity:", lo, hi, "outside:", n)


lice_weekly proposed mutually exclusive flags: {'ok': 6813, 'unusual_high': 138}
LOCALITY | FLAGS
10110 {'ok': 197}
10262 {'ok': 164}
10292 {'ok': 153, 'unusual_high': 1}
10324 {'ok': 132, 'unusual_high': 5}
10328 {'ok': 165}
11290 {'ok': 156, 'unusual_high': 2}
11312 {'ok': 178, 'unusual_high': 2}
11350 {'ok': 180}
11405 {'ok': 138}
11800 {'ok': 190, 'unusual_high': 1}
11922 {'ok': 207}
12244 {'ok': 181, 'unusual_high': 3}
12652 {'ok': 136, 'unusual_high': 3}
12660 {'ok': 201, 'unusual_high': 2}
12714 {'ok': 199}
12855 {'ok': 152, 'unusual_high': 4}
12897 {'ok': 192, 'unusual_high': 5}
13226 {'ok': 164}
13229 {'ok': 215, 'unusual_high': 8}
13870 {'ok': 190, 'unusual_high': 19}
20316 {'ok': 185, 'unusual_high': 1}
20559 {'ok': 150, 'unusual_high': 6}
22095 {'ok': 208, 'unusual_high': 11}
26955 {'ok': 154}
29816 {'ok': 151, 'unusual_high': 1}
30517 {'ok': 119, 'unusual_high': 7}
31497 {'ok': 133, 'unusual_high': 5}
31557 {'ok': 175, 'unusual_high': 3}
31757 {'ok': 195, 'unusual_high': 6

## Text, category and identity evidence for handoff

Distinct strings are shown with repr(), including metadata fields, to expose whitespace and Unicode.
Treatment counts do not establish that an absent treatment report means no intervention.
M2 kept latest metadata (Type 1), so historical naming/identity changes cannot be ruled out by this snapshot.
Egemen must perform and document his own six-family category/identity investigation and mappings.


In [6]:
print(json.dumps(p["category_inventory_for_egemen"], ensure_ascii=False, indent=2))
for column, values in p["text_inventory_for_egemen"].items():
    print("\nDISTINCT", column, "count", len(values))
    for value in values:
        print(value)
print("\nTreatment coverage (locality_no, weekly rows, first year, last year):")
for row in p["treatment_coverage_for_egemen"]:
    print(row)


{
  "treatment_category": [
    [
      "'combination'",
      34
    ],
    [
      "'medicinal'",
      164
    ],
    [
      "'non_medicinal'",
      691
    ]
  ],
  "treatment_type": [
    [
      "None",
      34
    ],
    [
      "'ANNEN_BEHANDLING'",
      55
    ],
    [
      "'BADEBEHANDLING'",
      28
    ],
    [
      "'Bath'",
      21
    ],
    [
      "'FERSKVANNSBEHANDLING'",
      45
    ],
    [
      "'FORBEHANDLING'",
      43
    ],
    [
      "'InFeed'",
      72
    ],
    [
      "'MEKANISK_BEHANDLING'",
      458
    ],
    [
      "'TERMISK_BEHANDLING'",
      133
    ]
  ],
  "substance_name": [
    [
      "None",
      725
    ],
    [
      "'Annet virkestoff'",
      5
    ],
    [
      "'Azamethiphos'",
      37
    ],
    [
      "'Deltamethrin'",
      3
    ],
    [
      "'Diflubenzuron'",
      12
    ],
    [
      "'Emamectin benzoat'",
      99
    ],
    [
      "'Hydrogenperoksid'",
      2
    ],
    [
      "'Other'",
      6
    ]
  

## Read-only guarantee and interpretation limits

The notebook changes no table. Historical missingness is heterogeneous by locality and year;
MCAR is not established. Activity-dependent reporting (potential MNAR) is a hypothesis, not a proven cause.
Future placeholders are structural absence, not sensor downtime. No gaps are filled and no causal or
prediction analysis is performed. Category/identity decisions and validate.py belong to Egemen's next stage.


In [7]:
print("M2 hashes unchanged:", snapshot(con) == before)
if snapshot(con) != before:
    raise RuntimeError("M2 changed during read-only profiling")
print("Profile complete; category/identity interpretation remains for Egemen.")
con.close()


M2 hashes unchanged: True
Profile complete; category/identity interpretation remains for Egemen.
